# Nonword phonological neighbourhood surprisal and BLICK

Both measures are computed per nonword from its phones, then averaged over each list.

## Phonological neighbourhood surprisal

The lexicon is every CMUdict pronunciation $w$ (stress removed). Its weight is the word's SUBTLEX-US frequency,
$f(w) = \text{count} + 1$, split evenly over the word's pronunciations, and $P_{\text{lex}}(w) = f(w) / \sum_v f(v)$.

$D(x, w)$ is the phone edit distance (Levenshtein over phones, so `SH` counts as one symbol), and $d$ is that distance
normalised by the longer of the two lengths:
$$d(x, w) = \frac{D(x, w)}{\max(|x|, |w|)}$$
A pronunciation identical to $x$ ($D = 0$) is left out.

Surprisal (nats), where higher means farther from every frequent word:
$$S_\beta(x) = -\ln \sum_{w \ne x} P_{\text{lex}}(w)\, e^{-\beta\, d(x, w)}$$

Entropy, where low means one neighbour dominates:
$$P_\beta(w \mid x) = \frac{f(w)\, e^{-\beta d(x, w)}}{\sum_{v \ne x} f(v)\, e^{-\beta d(x, v)}}, \qquad
H_\beta(x) = -\sum_{w \ne x} P_\beta(w \mid x) \ln P_\beta(w \mid x)$$

PLD20 is the mean raw distance $D$ to the 20 nearest pronunciations. Here $\beta = 20$; the pipeline also writes $\beta = 10$.

## BLICK

BLICK (Hayes & Wilson 2008; python-BLICK 0.2.12) scores a phone string with a maximum-entropy phonotactic grammar
of English. Each constraint $c$ has a weight $w_c$, and $n_c(x)$ counts how often $x$ violates it:
$$\text{BLICK}(x) = \sum_c w_c\, n_c(x), \qquad P(x) \propto e^{-\text{BLICK}(x)}$$
So BLICK is a surprisal up to a constant: 0 is fully English-like, and higher is less so.

Every vowel is given primary stress, because the stress of a nonword is unknown. This inflates the scores of words
with more than one syllable: a legal three-syllable nonword can score above 20.

List score: the mean over the list's nonwords, for each measure.

## Setup (once)

```bash
pip install -e ".[notebooks]"                   # includes python-BLICK 0.2.12
python -c "import nltk; nltk.download('cmudict')"
linguistic fetch subtlex_us
```

## Input

You can give either of two inputs:
- **`NONWORD_LISTS`**: lists of nonwords, each written as ARPAbet phones (`"bnick": "B N IH K"`). The notebook writes
  Audacity label files from them, with a nominal 0.1 s per phone, so the pipeline can read them as hand labels.
- **`STIMULI`**: a stimuli CSV whose nonwords already have hand phone labels (`words_labels` + `phones_labels`, or
  `labels`). Setting it overrides `NONWORD_LISTS`.

The example lists below are made up for this notebook, phone by phone, and none of them is in CMUdict. They are
graded by design:
1. legal short nonwords;
2. legal nonwords with clusters;
3. marginal onsets (*bn*, *tl*, *shm*);
4. illegal onsets (*ng*, *lb*, *rt*).

All are one syllable, to avoid the BLICK stress issue.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

STIMULI = None          # or a CSV with hand phone labels, e.g. ROOT / "my_nonwords.csv"
NONWORD_LISTS = {
    "legal_short":    {"mip": "M IH P", "fot": "F AA T", "bep": "B EH P", "dav": "D AE V",
                       "keb": "K EH B", "gup": "G AH P", "pob": "P AA B", "sheg": "SH EH G"},
    "legal_cluster":  {"spleem": "S P L IY M", "thrulk": "TH R AH L K", "skwemp": "S K W EH M P",
                       "straft": "S T R AE F T", "frownd": "F R OW N D", "blant": "B L AE N T",
                       "smelb": "S M EH L B", "drol": "D R AA L"},
    "marginal_onset": {"bnick": "B N IH K", "pwet": "P W EH T", "tlap": "T L AE P", "dlim": "D L IH M",
                       "vrop": "V R AA P", "shmeb": "SH M EH B", "thwup": "TH W AH P", "bwip": "B W IH P"},
    "illegal_onset":  {"ngop": "NG AA P", "lbik": "L B IH K", "rtap": "R T AE P", "mgot": "M G AA T",
                       "tfub": "T F AH B", "dbal": "D B AE L", "ngish": "NG IH SH", "rnup": "R N AH P"},
}
BETA = 20
OUT = ROOT / "out" / "nonword_phonological"


def write_inputs(lists, out, step=0.1):
    """One words + one phones Audacity file per list; returns the stimuli CSV."""
    (out / "labels").mkdir(parents=True, exist_ok=True)
    rows = []
    for sid, items in lists.items():
        t, wl, pl = 0.0, [], []
        for word, phones in items.items():
            start = t
            for p in phones.split():
                pl.append(f"{t:.3f}\t{t + step:.3f}\t{p}")
                t += step
            wl.append(f"{start:.3f}\t{t:.3f}\t{word}")
            t += step                                       # pause between nonwords
        (out / "labels" / f"{sid}_words.txt").write_text("\n".join(wl) + "\n")
        (out / "labels" / f"{sid}_phones.txt").write_text("\n".join(pl) + "\n")
        rows.append({"stimulus_id": sid, "condition": "nonword", "text": " ".join(items),
                     "words_labels": f"labels/{sid}_words.txt", "phones_labels": f"labels/{sid}_phones.txt"})
    pd.DataFrame(rows).to_csv(out / "stimuli.csv", index=False)
    return out / "stimuli.csv"


if STIMULI is None:
    STIMULI = write_inputs(NONWORD_LISTS, OUT)
pd.read_csv(STIMULI)[["stimulus_id", "condition", "text"]]

,stimulus_id,condition,text
0,legal_short,nonword,mip fot bep dav keb gup pob sheg
1,legal_cluster,nonword,spleem thrulk skwemp straft frownd blant smelb...
2,marginal_onset,nonword,bnick pwet tlap dlim vrop shmeb thwup bwip
3,illegal_onset,nonword,ngop lbik rtap mgot tfub dbal ngish rnup


## Step 1: run the pipeline

In [2]:
from linguistic.assemble import run_metrics
from linguistic.io import load_stimuli
from linguistic.normalize import Lexicon, build_units
from linguistic.registry import Context, select

lexicon = Lexicon()
stimuli = load_stimuli(STIMULI)
units = build_units(stimuli, lexicon)
res = run_metrics(Context(stimuli, units.words, units.phones, lexicon),
                  select(["phonological_neighbourhood", "blick_word"]))

SCORES = ["phone_pld20", f"phone_nbhd_surprisal_b{BETA}", f"phone_nbhd_entropy_b{BETA}", "blick_word"]
words = res.words[res.words["phones"] != ""]
print(f"{len(words)} of {len(res.words)} nonwords have phones")
words[["stimulus_id", "word", "phones", *SCORES]].head()

32 of 32 nonwords have phones


,stimulus_id,word,phones,phone_pld20,phone_nbhd_surprisal_b20,phone_nbhd_entropy_b20,blick_word
0,legal_short,mip,M IH P,1.0,13.465966,3.030034,1.265
1,legal_short,fot,F AA T,1.0,11.152739,1.691700,0.776
2,legal_short,bep,B EH P,1.0,13.608404,3.070583,1.381
3,legal_short,dav,D AE V,1.0,11.563390,0.697475,2.717
4,legal_short,keb,K EH B,1.0,13.895397,1.850107,1.782


## Step 2: recompute from scratch

The same formulas with CMUdict, the SUBTLEX-US file, `rapidfuzz` and `blick` alone, checked against step 1.

In [3]:
import re

import nltk
from blick import BlickLoader
from rapidfuzz.distance import Levenshtein
from rapidfuzz.process import cdist

from linguistic import resources

# lexicon: every CMUdict pronunciation, stress removed; f = SUBTLEX-US count + 1, split over the word's pronunciations
subtlex = pd.read_csv(resources.path("subtlex_us"), sep="\t", keep_default_na=False, na_values=[""])
count = dict(zip(subtlex["Word"].astype(str).str.lower(), subtlex["FREQcount"]))
prons, f = [], []
for word, ps in nltk.corpus.cmudict.dict().items():
    for p in ps:
        prons.append(tuple(re.sub(r"\d", "", ph) for ph in p))
        f.append((count.get(word.lower(), 0) + 1) / len(ps))
f = np.array(f)
p_lex = f / f.sum()

# phones -> one character each, so the edit distance counts phones, not letters
code_of = {}
encode = lambda ph: "".join(code_of.setdefault(p, chr(0xE000 + len(code_of))) for p in ph)
lex = [encode(p) for p in prons]
lex_len = np.array([len(p) for p in prons])


def neighbourhood(phones, beta=BETA):
    x = encode(phones)
    D = cdist([x], lex, scorer=Levenshtein.distance, workers=-1)[0].astype(float)
    keep = D > 0                                           # leave identical pronunciations out
    D, d = D[keep], D[keep] / np.maximum(len(x), lex_len[keep])
    surprisal = -np.log((p_lex[keep] * np.exp(-beta * d)).sum())
    w = f[keep] * np.exp(-beta * d)
    p = w / w.sum()
    entropy = -(p[p > 0] * np.log(p[p > 0])).sum()
    return {"phone_pld20": np.sort(D)[:20].mean(),
            f"phone_nbhd_surprisal_b{beta}": surprisal, f"phone_nbhd_entropy_b{beta}": entropy}


VOWELS = {"AA", "AE", "AH", "AO", "AW", "AY", "EH", "ER", "EY", "IH", "IY", "OW", "OY", "UH", "UW"}
blick = BlickLoader()


def blick_score(phones):
    return blick.assessWord(" ".join(p + "1" if p in VOWELS else p for p in phones))   # primary stress on vowels


scratch = pd.DataFrame([{"stimulus_id": r.stimulus_id, "word": r.word, "phones": r.phones,
                         **neighbourhood(r.phones.split()), "blick_word": blick_score(r.phones.split())}
                        for r in words.itertuples()])
for c in SCORES:
    print(f"{c:24s} max |scratch - pipeline| = {np.abs(scratch[c].to_numpy() - words[c].to_numpy()).max():.2e}")

phone_pld20              max |scratch - pipeline| = 0.00e+00
phone_nbhd_surprisal_b20 max |scratch - pipeline| = 1.78e-15
phone_nbhd_entropy_b20   max |scratch - pipeline| = 1.78e-15
blick_word               max |scratch - pipeline| = 0.00e+00


## Per nonword

In [4]:
scratch.sort_values(f"phone_nbhd_surprisal_b{BETA}").round(2)

,stimulus_id,word,phones,phone_pld20,phone_nbhd_surprisal_b20,phone_nbhd_entropy_b20,blick_word
27,illegal_onset,mgot,M G AA T,1.80,10.64,0.26,12.65
1,legal_short,fot,F AA T,1.00,11.15,1.69,0.78
3,legal_short,dav,D AE V,1.00,11.56,0.70,2.72
12,legal_cluster,frownd,F R OW N D,1.55,11.57,1.00,1.91
5,legal_short,gup,G AH P,1.00,12.11,0.75,2.90
14,legal_cluster,smelb,S M EH L B,1.65,12.83,1.72,2.94
13,legal_cluster,blant,B L AE N T,1.10,13.05,3.50,0.12
17,marginal_onset,pwet,P W EH T,1.45,13.16,3.27,5.22
0,legal_short,mip,M IH P,1.00,13.47,3.03,1.26
15,legal_cluster,drol,D R AA L,1.20,13.59,2.36,0.40


## List level: mean over the list's nonwords

In [5]:
lists = scratch.groupby("stimulus_id", sort=False)[SCORES].mean()
lists.insert(0, "n_nonwords", scratch.groupby("stimulus_id", sort=False).size())
lists.round(2)

,n_nonwords,phone_pld20,phone_nbhd_surprisal_b20,phone_nbhd_entropy_b20,blick_word
stimulus_id,,,,,
legal_short,8,1.00,13.04,2.13,1.87
legal_cluster,8,1.67,13.71,3.05,1.88
marginal_onset,8,1.54,14.21,3.12,5.86
illegal_onset,8,1.60,14.03,2.55,13.88


## Examples: lists with the highest and lowest mean surprisal and BLICK

In [6]:
text = pd.read_csv(STIMULI).set_index("stimulus_id")["text"]


def show(col, unit):
    ranked = lists.sort_values(col, ascending=False)
    print(col)
    for label, ids in [("  High (least English-like)", ranked.index[:2]),
                       ("  Low (most English-like)", ranked.index[::-1][:2])]:
        print(label)
        for sid in ids:
            print(f"    {ranked.loc[sid, col]:6.2f} {unit}  {sid}: {text[sid]}")
    print()


show(f"phone_nbhd_surprisal_b{BETA}", "nats")
show("blick_word", "    ")

phone_nbhd_surprisal_b20
  High (least English-like)
     14.21 nats  marginal_onset: bnick pwet tlap dlim vrop shmeb thwup bwip
     14.03 nats  illegal_onset: ngop lbik rtap mgot tfub dbal ngish rnup
  Low (most English-like)
     13.04 nats  legal_short: mip fot bep dav keb gup pob sheg
     13.71 nats  legal_cluster: spleem thrulk skwemp straft frownd blant smelb drol

blick_word
  High (least English-like)
     13.88       illegal_onset: ngop lbik rtap mgot tfub dbal ngish rnup
      5.86       marginal_onset: bnick pwet tlap dlim vrop shmeb thwup bwip
  Low (most English-like)
      1.87       legal_short: mip fot bep dav keb gup pob sheg
      1.88       legal_cluster: spleem thrulk skwemp straft frownd blant smelb drol



## Neighbourhood surprisal vs BLICK

In [7]:
from scipy.stats import spearmanr

scratch["n_phones"] = scratch["phones"].str.split().str.len()
for c in ["phone_pld20", f"phone_nbhd_surprisal_b{BETA}", f"phone_nbhd_entropy_b{BETA}", "n_phones"]:
    rho, p = spearmanr(scratch[c], scratch["blick_word"])
    print(f"{c:24s} Spearman rho with BLICK = {rho:5.2f}  (p = {p:.3g}, n = {len(scratch)} nonwords)")

phone_pld20              Spearman rho with BLICK =  0.38  (p = 0.0326, n = 32 nonwords)
phone_nbhd_surprisal_b20 Spearman rho with BLICK =  0.47  (p = 0.00685, n = 32 nonwords)
phone_nbhd_entropy_b20   Spearman rho with BLICK =  0.09  (p = 0.635, n = 32 nonwords)
n_phones                 Spearman rho with BLICK =  0.02  (p = 0.929, n = 32 nonwords)


The two measure different things:
- **BLICK** asks whether the sound sequence is legal in English.
- **Neighbourhood surprisal** asks how close the nonword is to real, frequent words.

A legal nonword can still be far from every word. An illegal one can be one phone from a common word: *mgot* has
an illegal onset (BLICK 12.7) but is *got* with an extra phone, so it has the lowest neighbourhood surprisal here.
In sets that mix short and long nonwords, both measures also grow with length, so control for `n_phones`.